<a href="https://colab.research.google.com/github/aayushijain28/aayushijain_ml2_lab/blob/main/Experiment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ----------------------------------------
# FOIL Algorithm - Simple Implementation
# Target: GrandDaughter(x, y)
# ----------------------------------------

# Positive examples
positive_examples = {
    ("Alice", "John"),
    ("Mary", "Robert"),
    ("Susan", "David")
}

# Negative examples
negative_examples = {
    ("John", "Alice"),
    ("Robert", "Mary"),
    ("David", "Susan"),
    ("Alice", "Robert")
}


# ----------------------------------------
# Background knowledge
# ----------------------------------------

Female = {
    "Alice",
    "Mary",
    "Susan"
}

Father = {
    ("John", "Alice"),
    ("Robert", "Mary"),
    ("David", "Susan"),

    # More relationships
    ("Tom", "John"),
    ("George", "Robert"),
    ("Paul", "David")
}


# ----------------------------------------
# Evaluate a literal
# ----------------------------------------

def covers(example, rule):
    """
    Checks whether a rule covers an example.

    example = (x, y)

    rule is a list of literals such as:
        [
            ("Father", "y", "z"),
            ("Father", "z", "x"),
            ("Female", "x")
        ]
    """

    x, y = example

    # Current variable bindings
    bindings = {
        "x": x,
        "y": y
    }

    for literal in rule:

        predicate = literal[0]

        # --------------------------------
        # Female(x) / Female(y) / Female(z)
        # --------------------------------
        if predicate == "Female":

            variable = literal[1]

            if variable not in bindings:
                return False

            value = bindings[variable]

            if value not in Female:
                return False

        # --------------------------------
        # Father(A, B)
        # --------------------------------
        elif predicate == "Father":

            var1 = literal[1]
            var2 = literal[2]

            # If both variables are already bound
            if var1 in bindings and var2 in bindings:

                if (bindings[var1], bindings[var2]) not in Father:
                    return False

            # If first variable is known but second isn't
            elif var1 in bindings:

                value1 = bindings[var1]

                found = False

                for father, child in Father:
                    if father == value1:
                        bindings[var2] = child
                        found = True
                        break

                if not found:
                    return False

            # If second variable is known but first isn't
            elif var2 in bindings:

                value2 = bindings[var2]

                found = False

                for father, child in Father:
                    if child == value2:
                        bindings[var1] = father
                        found = True
                        break

                if not found:
                    return False

            else:
                return False

    return True


# ----------------------------------------
# Count positive and negative examples
# ----------------------------------------

def count_coverage(rule):

    positive_count = 0
    negative_count = 0

    for example in positive_examples:
        if covers(example, rule):
            positive_count += 1

    for example in negative_examples:
        if covers(example, rule):
            negative_count += 1

    return positive_count, negative_count


# ----------------------------------------
# Candidate literals
# ----------------------------------------

candidate_literals = [

    ("Female", "x"),
    ("Female", "y"),

    ("Father", "x", "y"),
    ("Father", "y", "x"),

    ("Father", "x", "z"),
    ("Father", "z", "x"),

    ("Father", "y", "z"),
    ("Father", "z", "y")
]


# ----------------------------------------
# FOIL Gain
# ----------------------------------------

import math


def foil_gain(rule, candidate):

    old_rule = rule

    # Coverage before adding candidate
    p0, n0 = count_coverage(old_rule)

    # Coverage after adding candidate
    new_rule = old_rule + [candidate]

    p1, n1 = count_coverage(new_rule)

    if p1 == 0:
        return -float("inf")

    if p0 + n0 == 0:
        return -float("inf")

    old_probability = p0 / (p0 + n0)
    new_probability = p1 / (p1 + n1)

    if old_probability == 0 or new_probability == 0:
        return -float("inf")

    gain = p1 * (
        math.log2(new_probability)
        - math.log2(old_probability)
    )

    return gain


# ----------------------------------------
# Find the best literal
# ----------------------------------------

def find_best_literal(rule, candidates):

    best_literal = None
    best_gain = -float("inf")

    for candidate in candidates:

        gain = foil_gain(rule, candidate)

        print(
            "Candidate:",
            candidate,
            "Gain:",
            gain
        )

        if gain > best_gain:
            best_gain = gain
            best_literal = candidate

    return best_literal


# ----------------------------------------
# FOIL main algorithm
# ----------------------------------------

def FOIL():

    remaining_positive = set(positive_examples)

    learned_rules = []

    while remaining_positive:

        print("\n----------------------------")
        print("Starting a new rule")
        print("----------------------------")

        rule = []

        candidates = candidate_literals.copy()

        # --------------------------------
        # Inner loop
        # --------------------------------

        while True:

            positive_count, negative_count = count_coverage(rule)

            print("\nCurrent Rule:", rule)
            print("Positive covered:", positive_count)
            print("Negative covered:", negative_count)

            # Stop when no negative examples are covered
            if negative_count == 0:
                break

            # Find best candidate
            best_literal = find_best_literal(
                rule,
                candidates
            )

            if best_literal is None:
                break

            print("Selected:", best_literal)

            rule.append(best_literal)

            # Remove selected literal
            candidates.remove(best_literal)

            # --------------------------------
            # Generate new variables
            # --------------------------------

            if "z" in str(best_literal):

                candidates.extend([
                    ("Female", "z"),
                    ("Father", "z", "w"),
                    ("Father", "w", "z")
                ])

        # --------------------------------
        # Store learned rule
        # --------------------------------

        learned_rules.append(rule)

        print("\nLearned Rule:")
        print("GrandDaughter(x,y) <-", rule)

        # --------------------------------
        # Remove covered positive examples
        # --------------------------------

        covered = set()

        for example in remaining_positive:

            if covers(example, rule):
                covered.add(example)

        remaining_positive -= covered

        print("Removed positive examples:", covered)
        print("Remaining positive examples:",
              remaining_positive)

    return learned_rules


# ----------------------------------------
# Run FOIL
# ----------------------------------------

rules = FOIL()

print("\n============================")
print("FINAL RULES")
print("============================")

for rule in rules:
    print("GrandDaughter(x,y) <-", rule)



----------------------------
Starting a new rule
----------------------------

Current Rule: []
Positive covered: 3
Negative covered: 4
Candidate: ('Female', 'x') Gain: 2.422064766172813
Candidate: ('Female', 'y') Gain: -inf
Candidate: ('Father', 'x', 'y') Gain: -inf
Candidate: ('Father', 'y', 'x') Gain: 3.6671772640093443
Candidate: ('Father', 'x', 'z') Gain: -inf
Candidate: ('Father', 'z', 'x') Gain: 0.0
Candidate: ('Father', 'y', 'z') Gain: 2.422064766172813
Candidate: ('Father', 'z', 'y') Gain: 0.0
Selected: ('Father', 'y', 'x')

Current Rule: [('Father', 'y', 'x')]
Positive covered: 3
Negative covered: 0

Learned Rule:
GrandDaughter(x,y) <- [('Father', 'y', 'x')]
Removed positive examples: {('Susan', 'David'), ('Mary', 'Robert'), ('Alice', 'John')}
Remaining positive examples: set()

FINAL RULES
GrandDaughter(x,y) <- [('Father', 'y', 'x')]


In [ ]:
import pandas as pd
import math
import os

# ============================================================
# 1. LOAD GAME OF THRONES DATASET
# ============================================================

import kagglehub
path = kagglehub.dataset_download("mylesoneill/game-of-thrones")

# List files in the downloaded path to find the correct CSV file
print(f"Downloaded dataset to: {path}")
print(f"Files in directory: {os.listdir(path)}")

# Assuming the main CSV file is 'character-predictions.csv' or similar within the downloaded folder
# Let's try to find a suitable CSV file. A common one for this dataset is character-predictions.csv
csv_file_name = "character-predictions.csv" # Or 'character-predictions_pose.csv' if it exists
full_csv_path = os.path.join(path, csv_file_name)

# If the exact file name isn't found, iterate and pick the first CSV
if not os.path.exists(full_csv_path):
    for f in os.listdir(path):
        if f.endswith('.csv'):
            full_csv_path = os.path.join(path, f)
            print(f"Found CSV file: {f}")
            break
    else:
        raise FileNotFoundError("No CSV file found in the downloaded dataset directory.")

df = pd.read_csv(full_csv_path)

print("Dataset loaded")
print("Number of characters:", len(df))
print()


# ============================================================
# 2. CLEAN DATA
# ============================================================

# Convert names to strings and remove missing values
df["name"] = df["name"].fillna("").astype(str)
df["father"] = df["father"].fillna("").astype(str)
df["mother"] = df["mother"].fillna("").astype(str)

# Remove rows without a name
df = df[df["name"].str.strip() != ""]


# ============================================================
# 3. CREATE LOGICAL FACTS
#
# Female(X)
# Male(X)
# Father(X,Y)
# Mother(X,Y)
# ============================================================

female = set()
male = set()

father = set()
mother = set()

people = set(df["name"])


for _, row in df.iterrows():

    child = row["name"].strip()

    # male = 1 according to the Kaggle dataset
    if row["male"] == 1:
        male.add(child)
    else:
        female.add(child)

    # Father relation
    if row["father"].strip() != "":
        father_name = row["father"].strip()

        father.add((father_name, child))
        people.add(father_name)

    # Mother relation
    if row["mother"].strip() != "":
        mother_name = row["mother"].strip()

        mother.add((mother_name, child))
        people.add(mother_name)


print("People:", len(people))
print("Female facts:", len(female))
print("Male facts:", len(male))
print("Father facts:", len(father))
print("Mother facts:", len(mother))
print()


# ============================================================
# 4. GENERATE GRANDDAUGHTER POSITIVE EXAMPLES
#
# GrandDaughter(X,Y) means:
#
# X is a granddaughter of Y
#
# Example:
#
# Father(Y,Z)
# Father(Z,X)
# Female(X)
#
# Therefore:
#
# GrandDaughter(X,Y)
# ============================================================

def generate_granddaughter_examples():

    positives = set()

    # --------------------------------------------
    # Father -> Father -> Female
    # --------------------------------------------

    for grandfather, parent in father:

        for parent2, child in father:

            if parent == parent2:

                # child must be female
                if child in female:

                    positives.add(
                        (child, grandfather)
                    )

    # --------------------------------------------
    # Father -> Mother -> Female
    # --------------------------------------------

    for grandfather, parent in father:

        for mother_person, child in mother:  # Renamed 'mother' to 'mother_person'

            if parent == mother_person:

                if child in female:

                    positives.add(
                        (child, grandfather)
                    )

    return positives


positive_examples = generate_granddaughter_examples()


print("Positive GrandDaughter examples:",
      len(positive_examples))


# ============================================================
# 5. GENERATE NEGATIVE EXAMPLES
#
# We generate random possible pairs that are NOT known
# to be GrandDaughter relationships.
# ============================================================

negative_examples = set()

people_list = list(people)

for x in people_list:

    for y in people_list:

        if x == y:
            continue

        if (x, y) not in positive_examples:

            negative_examples.add((x, y))


# Keep negative examples manageable
negative_examples = set(
    list(negative_examples)
    [:max(len(positive_examples) * 3, 100)]
)


print("Negative examples:",
      len(negative_examples))
print()


# ============================================================
# 6. BACKGROUND KNOWLEDGE
# ============================================================

facts = {
    "Female": female,
    "Male": male,
    "Father": father,
    "Mother": mother
}


# ============================================================
# 7. VARIABLE BINDING
# ============================================================

def is_variable(value):

    return value in {
        "x",
        "y",
        "z",
        "w"
    }


# ============================================================
# 8. CHECK WHETHER A LITERAL CAN BE SATISFIED
# ============================================================

def satisfy_literal(literal, bindings):

    predicate = literal[0]
    args = literal[1:]

    relation = facts[predicate]

    # --------------------------------------------------------
    # Unary predicate
    #
    # Female(x)
    # Male(x)
    # --------------------------------------------------------

    if predicate in ["Female", "Male"]:

        variable = args[0]

        # Variable already bound
        if variable in bindings:

            value = bindings[variable]

            if value in relation:
                return [bindings]

            return []

        # Variable not bound
        results = []

        for value in relation:

            new_bindings = bindings.copy()

            new_bindings[variable] = value

            results.append(new_bindings)

        return results

    # --------------------------------------------------------
    # Binary predicate
    #
    # Father(x,y)
    # Mother(x,y)
    # --------------------------------------------------------

    var1 = args[0]
    var2 = args[1]

    results = []

    for value1, value2 in relation:

        new_bindings = bindings.copy()

        # Check first variable
        if var1 in new_bindings:

            if new_bindings[var1] != value1:
                continue

        else:

            new_bindings[var1] = value1

        # Check second variable
        if var2 in new_bindings:

            if new_bindings[var2] != value2:
                continue

        else:

            new_bindings[var2] = value2

        results.append(new_bindings)

    return results


# ============================================================
# 9. CHECK WHETHER A RULE COVERS AN EXAMPLE
# ============================================================

def covers(example, rule):

    x_value, y_value = example

    bindings = {
        "x": x_value,
        "y": y_value
    }

    bindings_list = [bindings]

    for literal in rule:

        new_bindings_list = []

        for binding in bindings_list:

            results = satisfy_literal(
                literal,
                binding
            )

            new_bindings_list.extend(results)

        bindings_list = new_bindings_list

        if not bindings_list:
            return False

    return len(bindings_list) > 0


# ============================================================
# 10. COUNT POSITIVE / NEGATIVE COVERAGE
# ============================================================

def coverage(rule):

    positive_count = 0
    negative_count = 0

    for example in positive_examples:

        if covers(example, rule):

            positive_count += 1

    for example in negative_examples:

        if covers(example, rule):

            negative_count += 1

    return positive_count, negative_count


# ============================================================
# 11. FOIL INFORMATION GAIN
# ============================================================

def foil_gain(rule, literal):

    old_positive, old_negative = coverage(rule)

    new_rule = rule + [literal]

    new_positive, new_negative = coverage(new_rule)

    # Candidate covers no positive examples
    if new_positive == 0:
        return -float("inf")

    # Old rule has no examples
    if old_positive + old_negative == 0:
        return -float("inf")

    old_probability = (
        old_positive /
        (old_positive + old_negative)
    )

    new_probability = (
        new_positive /
        (new_positive + new_negative)
    )

    if old_probability == 0:
        return -float("inf")

    gain = new_positive * (
        math.log2(new_probability)
        -
        math.log2(old_probability)
    )

    return gain


# ============================================================
# 12. GENERATE CANDIDATE LITERALS
# ============================================================

def generate_candidates(variables):

    candidates = []

    # Unary predicates
    for variable in variables:

        candidates.append(
            ("Female", variable)
        )

        candidates.append(
            ("Male", variable)
        )

    # Binary predicates
    for v1 in variables:

        for v2 in variables:

            if v1 == v2:
                continue

            candidates.append(
                ("Father", v1, v2)
            )

            candidates.append(
                ("Mother", v1, v2)
            )

    return candidates


# ============================================================
# 13. FIND BEST LITERAL
# ============================================================

def find_best_literal(rule, candidates):

    best_literal = None
    best_gain = -float("inf")

    print("\nCandidate literals:")

    for candidate in candidates:

        gain = foil_gain(
            rule,
            candidate
        )

        print(
            f"{candidate} -> Gain = {gain:.4f}"
            if gain != -float("inf")
            else
            f"{candidate} -> Gain = -inf"
        )

        if gain > best_gain:

            best_gain = gain
            best_literal = candidate

    return best_literal, best_gain


# ============================================================
# 14. PRINT RULE
# ============================================================

def print_rule(rule):

    if not rule:

        print("GrandDaughter(x,y)")

        return

    rule_text = []

    for literal in rule:

        if len(literal) == 2:

            rule_text.append(
                f"{literal[0]}({literal[1]})"
            )

        else:

            rule_text.append(
                f"{literal[0]}({literal[1]},{literal[2]})"
            )

    print(
        "GrandDaughter(x,y) <- "
        + " AND ".join(rule_text)
    )


# ============================================================
# 15. FOIL INNER LOOP
# ============================================================

def learn_one_rule(remaining_positive):

    rule = []

    variables = ["x", "y", "z", "w"]

    candidates = generate_candidates(
        variables
    )

    print("\n================================")
    print("STARTING NEW RULE")
    print("================================")

    while True:

        positive_count, negative_count = coverage(rule)

        print("\nCurrent rule:")
        print_rule(rule)

        print(
            "Positive coverage:",
            positive_count
        )

        print(
            "Negative coverage:",
            negative_count
        )

        # ----------------------------------------------------
        # Stop when no negative examples are covered
        # ----------------------------------------------------

        if negative_count == 0:

            print(
                "\nNo negative examples covered."
            )

            break

        # ----------------------------------------------------
        # Find best candidate
        # ----------------------------------------------------

        best_literal, best_gain = (
            find_best_literal(
                rule,
                candidates
            )
        )

        if best_literal is None:

            print(
                "No useful literal found."
            )

            break

        print(
            "\nSelected literal:",
            best_literal
        )

        print(
            "FOIL Gain:",
            best_gain
        )

        # ----------------------------------------------------
        # Add literal
        # ----------------------------------------------------

        rule.append(best_literal)

        candidates.remove(
            best_literal
        )

        # ----------------------------------------------------
        # Prevent excessively long rules
        # ----------------------------------------------------

        if len(rule) >= 5:

            break

    return rule


# ============================================================
# 16. MAIN FOIL ALGORITHM
# ============================================================

def FOIL():

    remaining_positive = set(
        positive_examples
    )

    learned_rules = []

    iteration = 1

    while remaining_positive:

        print("\n\n")
        print("################################")
        print(
            "OUTER ITERATION:",
            iteration
        )
        print("################################")

        print(
            "Remaining positive examples:",
            len(remaining_positive)
        )

        # ----------------------------------------------------
        # Learn one rule
        # ----------------------------------------------------

        rule = learn_one_rule(
            remaining_positive
        )

        learned_rules.append(rule)

        # ----------------------------------------------------
        # Find positive examples covered
        # ----------------------------------------------------

        covered_positive = set()

        for example in remaining_positive:

            if covers(example, rule):

                covered_positive.add(
                    example
                )

        # ----------------------------------------------------
        # Avoid infinite loop
        # ----------------------------------------------------

        if not covered_positive:

            print(
                "\nRule covers no remaining "
                "positive examples."
            )

            break

        # ----------------------------------------------------
        # Remove covered positives
        # ----------------------------------------------------

        remaining_positive -= (
            covered_positive
        )

        print("\nLearned rule:")

        print_rule(rule)

        print(
            "\nPositive examples removed:",
            len(covered_positive)
        )

        iteration += 1

    return learned_rules


# ============================================================
# 17. RUN FOIL
# ============================================================

print("\n\n")
print("============================================")
print("             RUNNING FOIL")
print("============================================")

rules = FOIL()


# ============================================================
# 18. FINAL RULES
# ============================================================

print("\n\n")
print("============================================")
print("              FINAL RULES")
print("============================================")

for i, rule in enumerate(rules, 1):

    print(f"\nRule {i}:")

    print_rule(rule)

Using Colab cache for faster access to the 'game-of-thrones' dataset.
Downloaded dataset to: /kaggle/input/game-of-thrones
Files in directory: ['character-predictions.csv', 'battles.csv', 'character-deaths.csv']
Dataset loaded
Number of characters: 1946

People: 1960
Female facts: 741
Male facts: 1205
Father facts: 26
Mother facts: 21

Positive GrandDaughter examples: 1
Negative examples: 100




             RUNNING FOIL



################################
OUTER ITERATION: 1
################################
Remaining positive examples: 1

STARTING NEW RULE

Current rule:
GrandDaughter(x,y)
Positive coverage: 1
Negative coverage: 100

Candidate literals:
('Female', 'x') -> Gain = 1.5289
('Male', 'x') -> Gain = -inf
('Female', 'y') -> Gain = -inf
('Male', 'y') -> Gain = 0.8002
('Female', 'z') -> Gain = 0.0000
('Male', 'z') -> Gain = 0.0000
('Female', 'w') -> Gain = 0.0000
('Male', 'w') -> Gain = 0.0000
('Father', 'x', 'y') -> Gain = -inf
('Mother', 'x', 'y') -> Gain = -inf
('Father', 'x